In [1]:
import pandas as pd
import numpy as np
import zipfile
import os
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

In [2]:
# ==============================================================================
# --- STEP 1 & 2: DATA INGESTION (UNZIP & LOAD) ---
# ==============================================================================
print("🏗️ --- PROJECT #20: BOOK RECOMMENDATION SYSTEM INITIALIZED --- 🏗️")

zip_path = "/content/Book Recommendation Dataset.zip"
extract_dir = "/content/book_data"

# Extracting the bricks for our library
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Loading the 3 core datasets
books = pd.read_csv(os.path.join(extract_dir, 'Books.csv'), low_memory=False)
users = pd.read_csv(os.path.join(extract_dir, 'Users.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'Ratings.csv'))

print(f"✅ Data Loaded. Ratings Shape: {ratings.shape}")

🏗️ --- PROJECT #20: BOOK RECOMMENDATION SYSTEM INITIALIZED --- 🏗️
✅ Data Loaded. Ratings Shape: (1149780, 3)


In [4]:
# ==============================================================================
# --- STEP 5: DATA CLEANING (FILTERING SPARSITY) ---
# CRITICAL: We only keep experienced users and popular books to reduce noise.
# ==============================================================================
print("\n🧹 STEP 5: Filtering Sparsity (Removing Noise)...")

# 1. Users who have rated more than 200 books (The "Critics")
x = ratings['User-ID'].value_counts() > 200
experienced_users = x[x].index
filtered_rating = ratings[ratings['User-ID'].isin(experienced_users)]

# Merge to get book titles for filtering and later use (consolidated step)
book_pivot_df = filtered_rating.merge(books, on='ISBN')

# 2. Books that have received at least 50 ratings (The "Populars")
y = book_pivot_df['Book-Title'].value_counts() >= 50
famous_books = y[y].index

# Filter the merged DataFrame (book_pivot_df) to get books by famous authors.
# This DataFrame will contain ratings from experienced users for famous books, along with book details.
final_df = book_pivot_df[book_pivot_df['Book-Title'].isin(famous_books)]

# To get `final_ratings` (ratings data only for famous books from experienced users),
# extract the unique ISBNs from `final_df` and use them to filter `filtered_rating`.
famous_book_isbns = final_df['ISBN'].unique()
final_ratings = filtered_rating[filtered_rating['ISBN'].isin(famous_book_isbns)]



🧹 STEP 5: Filtering Sparsity (Removing Noise)...


In [5]:
# ==============================================================================
# --- STEP 6: FEATURE ENGINEERING (USER-ITEM MATRIX) ---
# Transforming data into a pivot table where Rows=Books and Columns=Users
# ==============================================================================
print("\n📐 STEP 6: Building the User-Item Pivot Matrix...")

pt = final_df.pivot_table(index='Book-Title', columns='User-ID', values='Book-Rating')
pt.fillna(0, inplace=True) # Fill missing ratings with 0

# Convert to Sparse Matrix for performance optimization
book_sparse = csr_matrix(pt)


📐 STEP 6: Building the User-Item Pivot Matrix...


In [6]:
# ==============================================================================
# --- STEP 9: MODEL TRAINING (NEAREST NEIGHBORS) ---
# Using Cosine Similarity to find the closest vectors (books)
# ==============================================================================
print("\n🧠 STEP 9: Training the Recommendation Engine (KNN)...")

model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_sparse)


🧠 STEP 9: Training the Recommendation Engine (KNN)...


NearestNeighbors(algorithm='brute', metric='cosine')

In [7]:
# ==============================================================================
# --- STEP 10: PREDICTION FUNCTION ---
# ==============================================================================
def recommend(book_name):
    # Fetching index of the book
    try:
        index = np.where(pt.index == book_name)[0][0]
        distances, suggestions = model.kneighbors(pt.iloc[index, :].values.reshape(1, -1), n_neighbors=6)

        print(f"\n📚 Recommendations for '{book_name}':")
        for i in range(len(suggestions)):
            for j in suggestions[i]:
                if pt.index[j] != book_name: # Don't recommend the same book
                    print(f"👉 {pt.index[j]}")
    except IndexError:
        print(f"❌ Book '{book_name}' not found in the refined library.")

# --- TEST DRIVE ---
recommend('The Da Vinci Code')


📚 Recommendations for 'The Da Vinci Code':
👉 Angels &amp; Demons
👉 Touching Evil
👉 Saving Faith
👉 Middlesex: A Novel
👉 The Sweet Potato Queens' Book of Love


In [8]:
import joblib

# We are packing the equipment and materials at the construction site.
joblib.dump(model, 'book_model.pkl')
joblib.dump(pt, 'book_pivot.pkl')

print("✅ The Brain (Model) and Library Structure (Pivot Table) have been packaged and are ready for shipping!")

✅ The Brain (Model) and Library Structure (Pivot Table) have been packaged and are ready for shipping!


In [9]:
# In Colab, we only extract the necessary columns and save them.
mini_books = books[['Book-Title', 'Image-URL-L']].drop_duplicates('Book-Title')
mini_books.to_csv('books_mini.csv', index=False)
print("✅ Mini depo hazır! Şimdi 'books_mini.csv' dosyasını indir ve Hugging Face'e yükle.")

✅ Mini depo hazır! Şimdi 'books_mini.csv' dosyasını indir ve Hugging Face'e yükle.


# 📚 Project #20: Book Recommendation System (Collaborative Filtering)
**Architect:** Kemal Demirbaş 🏰🚀 | **Project Series:** 20 of 21

[![Hugging Face Space](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Live%20Demo-blue)](YOUR_HUGGINGFACE_LIVE_DEMO_LINK_HERE)
[![Python](https://img.shields.io/badge/Python-3.9+-yellow)](https://www.python.org/)
[![Algorithm](https://img.shields.io/badge/Algorithm-K--Nearest%20Neighbors%20(KNN)-green)](https://scikit-learn.org/stable/modules/neighbors.html)
[![Similarity Metric](https://img.shields.io/badge/Metric-Cosine%20Similarity-orange)](https://en.wikipedia.org/wiki/Cosine_similarity)

---

### 📚 Project Vision: The Architect of Taste
> *"We don't recommend books based on what they contain (NLP), but based on who loved them (Behavioral DNA). This is the power of the crowd."*

**Book Recommendation System** is an Unsupervised Machine Learning project designed to replicate the "People who liked X also liked Y" engine used by tech giants. By applying an Item-Based **Collaborative Filtering** architecture on over a million book ratings, this system identifies mathematically similar reading tastes across a sparse user-item matrix.

---

## 🧠 The Brain: Collaborative Filtering Architecture
This engine doesn't read the book titles; it analyzes the **vector space** created by human interaction:

1.  **Sparsity Management:** Crucially implemented the "200/50" rule—keeping only users with >200 ratings and books with >50 ratings to reduce noise and optimize performance.
2.  **User-Item Matrix:** Engineered a massive pivot table where Rows=Books, Columns=Users, and Values=Ratings (0-10).
3.  **K-Nearest Neighbors (KNN):** Deployed the KNN algorithm with `metric='cosine'` to find the nearest vectors in multi-dimensional space.
4.  **Cosine Similarity:** The mathematical proof of relevance:
    $$similarity = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{||\mathbf{A}|| ||\mathbf{B}||}$$

---

## 📈 Audit (Success Report)
Our test run proved the model's semantic relevance. By inputting 'The Da Vinci Code', the engine successfully recommended its contextually relevant predecessor, 'Angels & Demons', purely based on crowd-sourced voting patterns.

- **Sample Input:** `'The Da Vinci Code'`
- **Top Recommendation:** `👉 'Angels & Demons'` (Highly contextually relevant)

---

## 🛠️ Tech Stack
- **Machine Learning Engine:** Scikit-Learn (`NearestNeighbors`).
- **Data Engineering:** Pandas, NumPy, SciPy (`csr_matrix` for sparse optimization).
- **Deployment:** Streamlit & Hugging Face Spaces.

---

## 🚀 Live Book Discovery Platform
Try the recommendation engine in real-time. Enter a book title to discover what readers with similar tastes are loving:

👉 **[Hugging Face Live Demo: Book Recommendation System](https://huggingface.co/spaces/Ironside35/NextRead-Collaborative)**

---
*Next Stop: Project #21 - The GRAND FINALE (The Crown Jewel of the Series) 👑🏗️*